# 03 — Statistical Significance
Compare the QKC patch to the full-sky null distribution via Monte Carlo.

In [ ]:
import sys
sys.path.insert(0, '../src')
from qkc import load_map, extract_patch, patch_stats
from qkc import angular_power_spectrum, anomaly_score, monte_carlo_pvalue
from qkc import plot_power_spectrum, plot_null_distribution
import numpy as np

In [ ]:
cmap = load_map('../data/COM_CMB_IQU-smica_2048_R3.00_full.fits', field=0, nside_out=512)
ipix, vals = extract_patch(cmap, l_deg=-57.0, b_deg=-27.0, radius_deg=10.0)

In [ ]:
# Full-sky power spectrum vs ΛCDM theory
from qkc.stats import camb_theory_cl
ells, cl_obs = angular_power_spectrum(cmap, lmax=200)
cl_th = camb_theory_cl(lmax=200)
fig = plot_power_spectrum(ells, cl_obs, cl_th)
fig.savefig('../data/power_spectrum.png', dpi=150, bbox_inches='tight')

In [ ]:
# Monte Carlo significance of the patch mean
obs_mean = np.mean(vals[np.isfinite(vals)])
p_value, null_dist = monte_carlo_pvalue(
    cmap,
    observed_stat=obs_mean,
    stat_fn=np.mean,
    n_simulations=1000,
    radius_deg=10.0,
)
print(f'Observed mean: {obs_mean:.4f} μK')
print(f'Monte Carlo p-value: {p_value:.4f}')

In [ ]:
fig = plot_null_distribution(null_dist, obs_mean, p_value, stat_label='Patch Mean (μK)')
fig.savefig('../data/null_distribution.png', dpi=150, bbox_inches='tight')

In [ ]:
# KS test: patch values vs random same-size sample from full sky
import numpy as np
rng = np.random.default_rng(0)
ref_sample = cmap[rng.choice(len(cmap), size=len(ipix), replace=False)]
scores = anomaly_score(vals[np.isfinite(vals)], ref_sample[np.isfinite(ref_sample)])
import json
print(json.dumps(scores, indent=2))